# Exploratory Data Analysis – EIT Touch Classification

## Objectives

This notebook investigates three key observations from CNN training:

1. **Unstable validation loss** – spikes and non-convergence suggest overfitting or data issues
2. **Poor class separability under PCA** – motivates alternative projections (LDA, UMAP, etc.)
3. **Class 3 (Firm press) misclassified as Class 0 (No contact) / Class 1 (Light touch)** – investigate feature overlap

### Sections
- §1 Dataset overview and class balance
- §2 Feature-level statistics and distributions
- §3 Noise impact analysis (clean vs. noisy)
- §4 Dimensionality reduction and separability (PCA, LDA, UMAP, t-SNE)
- §5 Pairwise class overlap analysis (focus on Class 3 confusion)
- §6 Feature correlation and importance
- §7 Overfitting diagnostics and recommendations

In [ ]:
import sys
sys.path.insert(0, '../python')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix, classification_report
from scipy.stats import ks_2samp, wasserstein_distance

try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print("UMAP not installed – skipping UMAP cells. Install with: pip install umap-learn")

from data.load_dataset import load_mat_dataset, prepare_splits

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('colorblind')
%matplotlib inline

SEED = 42
DATA_PATH = Path('../data/eit_dataset.mat')

CLASS_NAMES = ['No contact', 'Light touch', 'Firm press', 'Point contact', 'Distributed']
N_CLASSES = len(CLASS_NAMES)

print("Setup complete.")

## §1 – Dataset Overview and Class Balance

In [ ]:
# Load both clean and noisy datasets
X_clean, y = load_mat_dataset(DATA_PATH, use_noisy=False)
X_noisy, _ = load_mat_dataset(DATA_PATH, use_noisy=True)

print(f"Dataset shape: {X_clean.shape[0]} samples × {X_clean.shape[1]} features")
print(f"Classes: {N_CLASSES} ({', '.join(CLASS_NAMES)})")
print(f"Label range: [{y.min()}, {y.max()}]")
print(f"\nClean X stats:  min={X_clean.min():.4f}, max={X_clean.max():.4f}, mean={X_clean.mean():.4f}")
print(f"Noisy X stats:  min={X_noisy.min():.4f}, max={X_noisy.max():.4f}, mean={X_noisy.mean():.4f}")

In [ ]:
# Class balance
unique, counts = np.unique(y, return_counts=True)
class_df = pd.DataFrame({'Class': CLASS_NAMES, 'Count': counts, 'Fraction': counts / len(y)})
display(class_df)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(CLASS_NAMES, counts, edgecolor='black', linewidth=0.5)
ax.axhline(len(y) / N_CLASSES, color='red', linestyle='--', label='Balanced (ideal)')
ax.set_ylabel('Sample Count')
ax.set_title('Class Distribution')
ax.legend()
for bar, c in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20, str(c), ha='center', fontsize=9)
plt.tight_layout()
plt.show()

# Imbalance ratio
imbalance_ratio = counts.max() / counts.min()
print(f"\nImbalance ratio (max/min): {imbalance_ratio:.2f}")

## §2 – Feature-Level Statistics and Distributions

Examine per-feature variance, skewness, and identify dead/constant features.

In [ ]:
# Per-feature statistics
feature_stats = pd.DataFrame({
    'mean': X_clean.mean(axis=0),
    'std': X_clean.std(axis=0),
    'min': X_clean.min(axis=0),
    'max': X_clean.max(axis=0),
    'range': X_clean.max(axis=0) - X_clean.min(axis=0),
})

print(f"Features with zero variance: {(feature_stats['std'] == 0).sum()}")
print(f"Features with very low variance (<1e-6): {(feature_stats['std'] < 1e-6).sum()}")
print(f"\nFeature range summary:")
display(feature_stats.describe().round(4))

In [ ]:
# Feature variance distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(feature_stats['std'], bins=50, edgecolor='black', linewidth=0.5)
axes[0].set_xlabel('Standard Deviation')
axes[0].set_ylabel('Number of Features')
axes[0].set_title('Feature Standard Deviation Distribution (Clean)')
axes[0].axvline(feature_stats['std'].median(), color='red', linestyle='--', label=f"Median={feature_stats['std'].median():.4f}")
axes[0].legend()

# Per-feature mean across classes
class_means = np.array([X_clean[y == c].mean(axis=0) for c in range(N_CLASSES)])
im = axes[1].imshow(class_means, aspect='auto', cmap='viridis')
axes[1].set_xlabel('Feature Index')
axes[1].set_ylabel('Class')
axes[1].set_yticks(range(N_CLASSES))
axes[1].set_yticklabels(CLASS_NAMES)
axes[1].set_title('Mean Feature Value per Class')
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.show()

In [ ]:
# Feature distributions for a random subset – per class
rng = np.random.default_rng(SEED)
sample_features = rng.choice(X_clean.shape[1], size=min(8, X_clean.shape[1]), replace=False)
sample_features.sort()

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for idx, (ax, feat) in enumerate(zip(axes.ravel(), sample_features)):
    for c in range(N_CLASSES):
        ax.hist(X_clean[y == c, feat], bins=30, alpha=0.5, label=CLASS_NAMES[c], density=True)
    ax.set_title(f'Feature {feat}')
    ax.set_xlabel('Value')
    if idx == 0:
        ax.legend(fontsize=7)

fig.suptitle('Per-Class Feature Distributions (Clean Data)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## §3 – Noise Impact Analysis

Compare clean vs. noisy to quantify per-feature noise perturbation and its effect on class boundaries.

In [ ]:
# Noise magnitude analysis
noise_delta = X_noisy - X_clean

print(f"Noise perturbation statistics:")
print(f"  Mean absolute perturbation: {np.abs(noise_delta).mean():.6f}")
print(f"  Std of perturbation: {noise_delta.std():.6f}")
print(f"  Max absolute perturbation: {np.abs(noise_delta).max():.6f}")

# Signal-to-noise ratio per feature
snr_per_feature = X_clean.std(axis=0) / (noise_delta.std(axis=0) + 1e-10)
print(f"\nSNR per feature: min={snr_per_feature.min():.2f}, median={np.median(snr_per_feature):.2f}, max={snr_per_feature.max():.2f}")
print(f"Features with SNR < 1 (noise > signal): {(snr_per_feature < 1).sum()}")
print(f"Features with SNR < 5: {(snr_per_feature < 5).sum()}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# SNR histogram
axes[0].hist(snr_per_feature, bins=50, edgecolor='black', linewidth=0.5)
axes[0].axvline(1, color='red', linestyle='--', label='SNR = 1')
axes[0].set_xlabel('SNR (signal std / noise std)')
axes[0].set_ylabel('Features')
axes[0].set_title('Per-Feature SNR Distribution')
axes[0].legend()

# Noise magnitude per class
noise_by_class = [np.abs(noise_delta[y == c]).mean() for c in range(N_CLASSES)]
axes[1].bar(CLASS_NAMES, noise_by_class, edgecolor='black', linewidth=0.5)
axes[1].set_ylabel('Mean |Noise|')
axes[1].set_title('Noise Magnitude by Class')
axes[1].tick_params(axis='x', rotation=15)

# Clean vs. noisy sample comparison
sample_idx = rng.choice(len(X_clean), 1)[0]
axes[2].plot(X_clean[sample_idx], label='Clean', alpha=0.8)
axes[2].plot(X_noisy[sample_idx], label='Noisy', alpha=0.8)
axes[2].set_xlabel('Feature Index')
axes[2].set_ylabel('Value')
axes[2].set_title(f'Sample {sample_idx} (Class: {CLASS_NAMES[y[sample_idx]]})')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Per-class Wasserstein distance between clean and noisy
# (measures how much noise shifts the distribution for each class)
print("Per-class distribution shift (Wasserstein distance, averaged over features):")
for c in range(N_CLASSES):
    mask = y == c
    dists = [wasserstein_distance(X_clean[mask, f], X_noisy[mask, f]) for f in range(X_clean.shape[1])]
    print(f"  Class {c} ({CLASS_NAMES[c]}): mean W-dist = {np.mean(dists):.6f}, max = {np.max(dists):.6f}")

## §4 – Dimensionality Reduction and Class Separability

Compare multiple projection methods to understand class structure:
- **PCA** – linear, unsupervised (known poor separability)
- **LDA** – linear, supervised (maximises class separation)
- **t-SNE** – non-linear, unsupervised
- **UMAP** – non-linear, can preserve global structure

In [ ]:
# Normalise for projections
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clean)
X_noisy_scaled = scaler.transform(X_noisy)

In [ ]:
# PCA – explained variance and 2D projection
pca = PCA(n_components=min(50, X_scaled.shape[1]))
X_pca_all = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Explained variance
cumvar = np.cumsum(pca.explained_variance_ratio_)
axes[0].plot(range(1, len(cumvar)+1), cumvar, 'o-', markersize=3)
axes[0].axhline(0.95, color='red', linestyle='--', label='95% variance')
axes[0].axhline(0.99, color='orange', linestyle='--', label='99% variance')
n_95 = np.argmax(cumvar >= 0.95) + 1
n_99 = np.argmax(cumvar >= 0.99) + 1
axes[0].axvline(n_95, color='red', linestyle=':', alpha=0.5)
axes[0].set_xlabel('Number of Components')
axes[0].set_ylabel('Cumulative Explained Variance')
axes[0].set_title('PCA – Explained Variance')
axes[0].legend()
print(f"Components for 95% variance: {n_95}")
print(f"Components for 99% variance: {n_99}")

# 2D scatter
for c in range(N_CLASSES):
    mask = y == c
    axes[1].scatter(X_pca_all[mask, 0], X_pca_all[mask, 1], label=CLASS_NAMES[c], alpha=0.4, s=10)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
axes[1].set_title('PCA – 2D Projection (Clean)')
axes[1].legend(markerscale=3)

plt.tight_layout()
plt.show()

In [ ]:
# LDA – supervised linear projection (maximises between-class / within-class variance)
lda = LDA(n_components=min(N_CLASSES - 1, X_scaled.shape[1]))
X_lda = lda.fit_transform(X_scaled, y)

print(f"LDA explained variance ratios: {lda.explained_variance_ratio_}")
print(f"Total LDA variance captured: {lda.explained_variance_ratio_.sum():.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# LDA 2D
for c in range(N_CLASSES):
    mask = y == c
    axes[0].scatter(X_lda[mask, 0], X_lda[mask, 1], label=CLASS_NAMES[c], alpha=0.4, s=10)
axes[0].set_xlabel(f'LD1 ({lda.explained_variance_ratio_[0]:.1%})')
axes[0].set_ylabel(f'LD2 ({lda.explained_variance_ratio_[1]:.1%})')
axes[0].set_title('LDA – 2D Projection (Clean)')
axes[0].legend(markerscale=3)

# LDA on noisy data
X_lda_noisy = lda.transform(X_noisy_scaled)
for c in range(N_CLASSES):
    mask = y == c
    axes[1].scatter(X_lda_noisy[mask, 0], X_lda_noisy[mask, 1], label=CLASS_NAMES[c], alpha=0.4, s=10)
axes[1].set_xlabel(f'LD1 ({lda.explained_variance_ratio_[0]:.1%})')
axes[1].set_ylabel(f'LD2 ({lda.explained_variance_ratio_[1]:.1%})')
axes[1].set_title('LDA – 2D Projection (Noisy)')
axes[1].legend(markerscale=3)

plt.tight_layout()
plt.show()

In [ ]:
# LDA separability metric: compute LDA classification accuracy as upper bound
from sklearn.model_selection import cross_val_score

lda_clf = LDA()
lda_scores = cross_val_score(lda_clf, X_scaled, y, cv=5, scoring='accuracy')
print(f"LDA 5-fold CV accuracy (clean): {lda_scores.mean():.4f} ± {lda_scores.std():.4f}")

lda_scores_noisy = cross_val_score(lda_clf, X_noisy_scaled, y, cv=5, scoring='accuracy')
print(f"LDA 5-fold CV accuracy (noisy): {lda_scores_noisy.mean():.4f} ± {lda_scores_noisy.std():.4f}")

print("\n→ If LDA accuracy is low, classes have significant feature overlap.")
print("→ If LDA accuracy is high but CNN struggles, the issue is model/training, not data.")

In [ ]:
# t-SNE (subsample for speed)
n_subsample = min(5000, len(X_scaled))
idx_sub = rng.choice(len(X_scaled), n_subsample, replace=False)

tsne = TSNE(n_components=2, perplexity=30, random_state=SEED, n_iter=1000)
X_tsne = tsne.fit_transform(X_scaled[idx_sub])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for c in range(N_CLASSES):
    mask = y[idx_sub] == c
    axes[0].scatter(X_tsne[mask, 0], X_tsne[mask, 1], label=CLASS_NAMES[c], alpha=0.5, s=10)
axes[0].set_title('t-SNE – Clean Data (perplexity=30)')
axes[0].legend(markerscale=3)

# t-SNE on noisy
tsne_noisy = TSNE(n_components=2, perplexity=30, random_state=SEED, n_iter=1000)
X_tsne_noisy = tsne_noisy.fit_transform(X_noisy_scaled[idx_sub])

for c in range(N_CLASSES):
    mask = y[idx_sub] == c
    axes[1].scatter(X_tsne_noisy[mask, 0], X_tsne_noisy[mask, 1], label=CLASS_NAMES[c], alpha=0.5, s=10)
axes[1].set_title('t-SNE – Noisy Data (perplexity=30)')
axes[1].legend(markerscale=3)

plt.tight_layout()
plt.show()

In [ ]:
# UMAP (if available)
if HAS_UMAP:
    reducer = umap.UMAP(n_components=2, random_state=SEED, n_neighbors=15, min_dist=0.1)
    X_umap = reducer.fit_transform(X_scaled[idx_sub])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for c in range(N_CLASSES):
        mask = y[idx_sub] == c
        axes[0].scatter(X_umap[mask, 0], X_umap[mask, 1], label=CLASS_NAMES[c], alpha=0.5, s=10)
    axes[0].set_title('UMAP – Clean Data')
    axes[0].legend(markerscale=3)

    X_umap_noisy = reducer.transform(X_noisy_scaled[idx_sub])
    for c in range(N_CLASSES):
        mask = y[idx_sub] == c
        axes[1].scatter(X_umap_noisy[mask, 0], X_umap_noisy[mask, 1], label=CLASS_NAMES[c], alpha=0.5, s=10)
    axes[1].set_title('UMAP – Noisy Data')
    axes[1].legend(markerscale=3)

    plt.tight_layout()
    plt.show()
else:
    print("Skipping UMAP – not installed.")

## §5 – Class 3 Confusion Analysis

Deep-dive into why Class 3 (Firm press) is confused with Class 0 (No contact) and Class 1 (Light touch).

In [ ]:
# Pairwise feature overlap: Class 0 vs 3, and Class 1 vs 3
def compute_class_overlap(X, y, class_a, class_b):
    """Compute per-feature overlap between two classes using KS-test."""
    mask_a = y == class_a
    mask_b = y == class_b
    n_features = X.shape[1]
    
    ks_stats = np.zeros(n_features)
    p_values = np.zeros(n_features)
    
    for f in range(n_features):
        stat, p = ks_2samp(X[mask_a, f], X[mask_b, f])
        ks_stats[f] = stat
        p_values[f] = p
    
    return ks_stats, p_values

# KS-test: higher statistic = more separable features
ks_03, p_03 = compute_class_overlap(X_clean, y, 0, 3)
ks_13, p_13 = compute_class_overlap(X_clean, y, 1, 3)
ks_01, p_01 = compute_class_overlap(X_clean, y, 0, 1)

print("Pairwise KS-test statistics (higher = more separable):")
print(f"  Class 0 vs 3: mean KS = {ks_03.mean():.4f}, features with p > 0.05: {(p_03 > 0.05).sum()} / {len(p_03)}")
print(f"  Class 1 vs 3: mean KS = {ks_13.mean():.4f}, features with p > 0.05: {(p_13 > 0.05).sum()} / {len(p_13)}")
print(f"  Class 0 vs 1: mean KS = {ks_01.mean():.4f}, features with p > 0.05: {(p_01 > 0.05).sum()} / {len(p_01)}")
print("\n→ Features with p > 0.05 cannot statistically distinguish between those classes.")

In [ ]:
# Visualize the overlap for the most and least separable features
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

pairs = [(0, 3, ks_03, 'Class 0 vs 3'), (1, 3, ks_13, 'Class 1 vs 3'), (0, 1, ks_01, 'Class 0 vs 1')]

for col, (ca, cb, ks, title) in enumerate(pairs):
    # Most separable feature
    best_feat = np.argmax(ks)
    ax = axes[0, col]
    ax.hist(X_clean[y == ca, best_feat], bins=40, alpha=0.6, label=CLASS_NAMES[ca], density=True)
    ax.hist(X_clean[y == cb, best_feat], bins=40, alpha=0.6, label=CLASS_NAMES[cb], density=True)
    ax.set_title(f'{title}\nMost separable (feat {best_feat}, KS={ks[best_feat]:.3f})')
    ax.legend()
    
    # Least separable feature
    worst_feat = np.argmin(ks)
    ax = axes[1, col]
    ax.hist(X_clean[y == ca, worst_feat], bins=40, alpha=0.6, label=CLASS_NAMES[ca], density=True)
    ax.hist(X_clean[y == cb, worst_feat], bins=40, alpha=0.6, label=CLASS_NAMES[cb], density=True)
    ax.set_title(f'{title}\nLeast separable (feat {worst_feat}, KS={ks[worst_feat]:.3f})')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# LDA projection focusing on the confused classes
# 3-class subset: No contact, Light touch, Firm press
mask_subset = np.isin(y, [0, 1, 3])
X_sub = X_scaled[mask_subset]
y_sub = y[mask_subset]

lda_sub = LDA(n_components=2)
X_lda_sub = lda_sub.fit_transform(X_sub, y_sub)

fig, ax = plt.subplots(figsize=(8, 6))
for c in [0, 1, 3]:
    mask = y_sub == c
    ax.scatter(X_lda_sub[mask, 0], X_lda_sub[mask, 1], label=CLASS_NAMES[c], alpha=0.5, s=15)
ax.set_xlabel('LD1')
ax.set_ylabel('LD2')
ax.set_title('LDA – Confused Classes Only (0, 1, 3)')
ax.legend(markerscale=3)
plt.tight_layout()
plt.show()

# LDA accuracy on this subset
lda_sub_scores = cross_val_score(LDA(), X_sub, y_sub, cv=5, scoring='accuracy')
print(f"LDA 5-fold accuracy on {{{CLASS_NAMES[0]}, {CLASS_NAMES[1]}, {CLASS_NAMES[3]}}}: {lda_sub_scores.mean():.4f} ± {lda_sub_scores.std():.4f}")

In [ ]:
# Class centroids and inter-class distances
centroids = np.array([X_scaled[y == c].mean(axis=0) for c in range(N_CLASSES)])

# Euclidean distance matrix between centroids
from scipy.spatial.distance import cdist
dist_matrix = cdist(centroids, centroids, metric='euclidean')

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(dist_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_title('Euclidean Distance Between Class Centroids')
plt.tight_layout()
plt.show()

print(f"\nDistance from Class 3 (Firm press) to:")
for c in range(N_CLASSES):
    if c != 3:
        print(f"  Class {c} ({CLASS_NAMES[c]}): {dist_matrix[3, c]:.4f}")

In [ ]:
# Within-class scatter vs. between-class distance for Class 3
within_scatter = np.array([X_scaled[y == c].std(axis=0).mean() for c in range(N_CLASSES)])

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(CLASS_NAMES, within_scatter, edgecolor='black', linewidth=0.5)
ax.set_ylabel('Mean Feature Std (within-class scatter)')
ax.set_title('Within-Class Scatter (lower = tighter clusters)')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

# Fisher criterion per-class pair
print("\nFisher Criterion (between-class dist / sum of within-class scatter):")
for i in range(N_CLASSES):
    for j in range(i+1, N_CLASSES):
        fisher = dist_matrix[i, j] / (within_scatter[i] + within_scatter[j] + 1e-10)
        marker = " ← LOW" if fisher < 1.0 else ""
        print(f"  {CLASS_NAMES[i]} vs {CLASS_NAMES[j]}: {fisher:.3f}{marker}")

## §6 – Feature Correlation and Importance

Identify redundant features and find which features best discriminate the confused classes.

In [ ]:
# Feature correlation matrix (subsample features if too many)
n_feat = X_clean.shape[1]
if n_feat > 50:
    # Show correlation of first 50 features
    corr = np.corrcoef(X_scaled[:, :50].T)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(corr, cmap='RdBu_r', center=0, vmin=-1, vmax=1, ax=ax)
    ax.set_title(f'Feature Correlation Matrix (first 50 of {n_feat} features)')
else:
    corr = np.corrcoef(X_scaled.T)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(corr, cmap='RdBu_r', center=0, vmin=-1, vmax=1, ax=ax)
    ax.set_title('Feature Correlation Matrix')

plt.tight_layout()
plt.show()

# Highly correlated feature pairs
upper_tri = np.triu(np.abs(corr), k=1)
high_corr = np.sum(upper_tri > 0.9)
print(f"Feature pairs with |correlation| > 0.9: {high_corr}")
print(f"Feature pairs with |correlation| > 0.95: {np.sum(upper_tri > 0.95)}")

In [ ]:
# Feature importance via Random Forest for discriminating Class 3
from sklearn.ensemble import RandomForestClassifier

# Binary problem: Class 3 vs. all others
y_binary_3 = (y == 3).astype(int)
rf = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
rf.fit(X_scaled, y_binary_3)

importances = rf.feature_importances_
top_k = 20
top_indices = np.argsort(importances)[-top_k:][::-1]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(top_k), importances[top_indices], edgecolor='black', linewidth=0.5)
ax.set_xticks(range(top_k))
ax.set_xticklabels([f'F{i}' for i in top_indices], rotation=45)
ax.set_xlabel('Feature Index')
ax.set_ylabel('Importance')
ax.set_title(f'Top-{top_k} Features for Distinguishing Class 3 (Firm press)')
plt.tight_layout()
plt.show()

print(f"\nTop-5 features: {top_indices[:5]}")
print(f"RF accuracy (Class 3 vs rest): {rf.score(X_scaled, y_binary_3):.4f}")

In [ ]:
# ANOVA F-statistic per feature (full multiclass)
from sklearn.feature_selection import f_classif

f_stats, p_vals = f_classif(X_scaled, y)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(range(len(f_stats)), f_stats, width=1.0)
axes[0].set_xlabel('Feature Index')
axes[0].set_ylabel('F-statistic')
axes[0].set_title('ANOVA F-statistic per Feature (multiclass)')

# Log p-values
axes[1].bar(range(len(p_vals)), -np.log10(p_vals + 1e-300), width=1.0, color='orange')
axes[1].axhline(-np.log10(0.05), color='red', linestyle='--', label='p=0.05')
axes[1].set_xlabel('Feature Index')
axes[1].set_ylabel('-log10(p-value)')
axes[1].set_title('ANOVA Significance per Feature')
axes[1].legend()

plt.tight_layout()
plt.show()

n_insignificant = (p_vals > 0.05).sum()
print(f"Features with p > 0.05 (not statistically useful): {n_insignificant} / {len(p_vals)}")

## §7 – Overfitting Diagnostics and Recommendations

Investigate potential causes of unstable validation loss and train/val gap.

In [ ]:
# Sample size analysis: is there enough data per class for the model capacity?
n_samples = len(y)
n_features_total = X_clean.shape[1]
n_params_cnn = 56000  # approx from architecture (~56k params)

print("=== Overfitting Risk Factors ===")
print(f"\n1. Sample-to-feature ratio: {n_samples / n_features_total:.1f}:1")
print(f"   Samples per class (avg): {n_samples / N_CLASSES:.0f}")
print(f"   CNN parameters: ~{n_params_cnn:,}")
print(f"   Samples-to-parameters ratio: {n_samples / n_params_cnn:.1f}:1")
print(f"   (Rule of thumb: want >10:1 for stable training)")

print(f"\n2. Feature dimensionality: {n_features_total}")
print(f"   Effective dimensions (95% PCA): {n_95}")
print(f"   → Data lives in ~{n_95}D subspace, model sees {n_features_total}D")

print(f"\n3. Class balance: imbalance ratio = {imbalance_ratio:.2f}")
min_class_idx = np.argmin(counts)
print(f"   Smallest class: {CLASS_NAMES[min_class_idx]} ({counts[min_class_idx]} samples)")

In [ ]:
# Learning curve simulation: subsample training data and measure gap
from sklearn.model_selection import learning_curve

# Quick proxy using LDA (fast to train)
train_sizes, train_scores, val_scores = learning_curve(
    LDA(), X_scaled, y, cv=5,
    train_sizes=np.linspace(0.1, 1.0, 10),
    scoring='accuracy', n_jobs=-1
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(train_sizes, train_scores.mean(axis=1), 'o-', label='Train', linewidth=2)
ax.fill_between(train_sizes, 
                train_scores.mean(axis=1) - train_scores.std(axis=1),
                train_scores.mean(axis=1) + train_scores.std(axis=1), alpha=0.1)
ax.plot(train_sizes, val_scores.mean(axis=1), 's--', label='Validation', linewidth=2)
ax.fill_between(train_sizes, 
                val_scores.mean(axis=1) - val_scores.std(axis=1),
                val_scores.mean(axis=1) + val_scores.std(axis=1), alpha=0.1)
ax.set_xlabel('Training Set Size')
ax.set_ylabel('Accuracy')
ax.set_title('Learning Curve (LDA proxy)')
ax.legend()
plt.tight_layout()
plt.show()

gap = train_scores.mean(axis=1)[-1] - val_scores.mean(axis=1)[-1]
print(f"\nFinal train-val gap: {gap:.4f}")
if gap > 0.05:
    print("→ Large gap suggests overfitting even for linear model.")
else:
    print("→ Small gap for LDA; CNN overfitting likely due to model complexity.")

In [ ]:
# Duplicate / near-duplicate detection
from scipy.spatial.distance import pdist, squareform

# Check for exact duplicates
_, unique_idx, unique_counts = np.unique(X_clean, axis=0, return_index=True, return_counts=True)
n_duplicates = len(X_clean) - len(unique_idx)
print(f"Exact duplicates in clean data: {n_duplicates} ({n_duplicates/len(X_clean)*100:.1f}%)")

if n_duplicates > 0:
    dup_mask = unique_counts > 1
    print(f"  Unique samples with copies: {dup_mask.sum()}")
    print(f"  Max copies of a single sample: {unique_counts.max()}")

# Near-duplicates (sample a subset for speed)
n_check = min(2000, len(X_scaled))
idx_check = rng.choice(len(X_scaled), n_check, replace=False)
dists = pdist(X_scaled[idx_check])
near_dup_threshold = 0.01
n_near_dup = (dists < near_dup_threshold).sum()
print(f"\nNear-duplicates (dist < {near_dup_threshold}): {n_near_dup} pairs in {n_check}-sample subset")

In [ ]:
# Validation loss instability: batch-level variance proxy
# Estimate per-sample prediction difficulty using cross-validated confidence
from sklearn.model_selection import cross_val_predict

# Use LDA to get per-sample predicted probabilities
y_probs = cross_val_predict(LDA(), X_scaled, y, cv=5, method='predict_proba')
y_confidence = y_probs[np.arange(len(y)), y]  # confidence on true class

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Per-class confidence
class_confidences = [y_confidence[y == c] for c in range(N_CLASSES)]
axes[0].boxplot(class_confidences, labels=CLASS_NAMES)
axes[0].set_ylabel('Prediction Confidence (true class)')
axes[0].set_title('LDA Confidence Distribution per Class')
axes[0].tick_params(axis='x', rotation=15)

# Histogram of low-confidence samples
axes[1].hist(y_confidence, bins=50, edgecolor='black', linewidth=0.5)
axes[1].axvline(0.5, color='red', linestyle='--', label='50% confidence')
axes[1].set_xlabel('Confidence')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Prediction Confidence')
axes[1].legend()

plt.tight_layout()
plt.show()

low_conf = (y_confidence < 0.5).sum()
print(f"\nSamples with < 50% confidence: {low_conf} ({low_conf/len(y)*100:.1f}%)")
print("\nPer-class breakdown of low-confidence samples:")
for c in range(N_CLASSES):
    n_low = ((y_confidence < 0.5) & (y == c)).sum()
    n_total = (y == c).sum()
    print(f"  Class {c} ({CLASS_NAMES[c]}): {n_low}/{n_total} ({n_low/n_total*100:.1f}%)")

## Summary and Recommendations

Run the cells above and interpret the results using this checklist:

### Unstable Validation Loss
- [ ] Check samples-to-parameters ratio (§7) – if < 10:1, model is too large
- [ ] Check for duplicates/near-duplicates leaking between splits
- [ ] Check per-batch variance from low-confidence samples  
- **Potential fixes**: increase dropout, reduce model size, add weight decay, use data augmentation, increase batch size

### Poor PCA Separability
- [ ] Compare PCA vs LDA accuracy (§4) – if LDA >> PCA embedding, class structure is non-isotropic
- [ ] Check effective dimensionality (§4) – if 95% variance requires many PCs, data is spread thin
- **Potential fixes**: use LDA features as input, apply supervised contrastive loss, or add class-conditional BatchNorm

### Class 3 → Class 0/1 Confusion
- [ ] Check Fisher criterion for Class 3 vs 0/1 (§5) – if < 1.0, fundamental overlap exists
- [ ] Check feature importance for Class 3 (§6) – are discriminative features noise-sensitive?
- [ ] Check centroid distances (§5) – is Class 3 geometrically close to 0/1?
- **Potential fixes**: class-weighted loss, SMOTE/oversampling for Class 3, feature engineering (e.g., pairwise voltage ratios), hierarchical classifier